# 01. Counter

**Tier:** Intermediate. **Why you'd use this:** You have a list of votes and need the winner and how many votes that name got. A dict and a loop work. `Counter` is a dict built for counts.

**After this lesson you can:** build a `Counter`; read `most_common`; explain why a missing key returns `0`.
About 25 minutes. **Uses:** [Dicts](../04-collections/03-dicts-lesson.ipynb)


## Quick review

1. Why is a generator expression single-pass?
2. What does a dict comprehension do when two items share a key?

<details><summary>Answers</summary>

The generator is spent after one walk. The later key overwrites the earlier value.
</details>


## The idea

`Counter` is a dict subclass that counts hashable items. Indexing a missing key returns `0` instead of raising. `most_common(n)` returns the top counts as `(item, count)` pairs. You can add and subtract counters when you merge tallies.

<details><summary>Explain more</summary>

`Counter(items)` walks once and stores counts. `Counter` still supports ordinary dict methods. The zero-default is the special behaviour that makes tallies short.
</details>


## See it work

Count votes and pick the winner.


In [1]:
from collections import Counter


def winner(votes):
    # 1. Count, then take the top pair.
    tallies = Counter(votes)
    name, _count = tallies.most_common(1)[0]
    return name, tallies[name]


print(winner(["a", "b", "a", "c", "a"]))
print(isinstance(Counter(), dict))

('a', 3)
True


## Check yourself

Answer these before you run the cell.

1. What does `Counter("aab")["z"]` print?
2. Is that the same as a plain dict missing key?


In [2]:
from collections import Counter

print(Counter("aab")["z"])
print(Counter("aab").most_common(1))

0
[('a', 2)]


<details><summary>Answers</summary>

It prints `0`. A plain dict would raise `KeyError`. `Counter` is still a dict subclass, with a different missing-key rule.
</details>


## Why it works this way

`Counter` subclasses `dict`, so it stores keys and values in a hash table like any other mapping. The difference is the missing-key path. A plain dict raises `KeyError`. `Counter` returns `0` for a missing key so tally code can write `counts[item] += 1` without a guard. That behaviour is deliberate for frequency work, not a general rule for every dict. You still get ordinary dict methods, iteration over keys, and updates. Deep dive: [How dict and set work](../../internals/04-dicts/01-hash-table-lesson.ipynb).

`most_common` sorts by count. Asking for the top item is one ordered pass over the stored keys, not a second walk of the original list. Building the counter already cost a linear walk over the input. Memory is one entry per distinct item. If almost every item is unique, a `Counter` is about as large as a dict of the same keys with integer values. That is the right cost when you need ranks. It is the wrong cost when you only needed a yes-or-no membership test.

The design goal is a dict that speaks the language of counts. Addition and subtraction of counters merge tallies. Negative counts are allowed after subtraction, which surprises people who expect counts to stay non-negative. Methods such as `elements` skip non-positive counts, so know which operation clamps and which does not. Treat `Counter` as a specialized dict, not as a bag that silently clamps at zero on every path.

Use evidence: `isinstance(Counter(), dict)` is true, and a missing key prints `0`. Those two facts explain most of the API. Reach for `Counter` when the question is "how many" or "which are most common." Reach for a plain dict when absence should be an error.


In [3]:
from collections import Counter

votes = ["a", "b", "a", "c", "a"]
tallies = Counter(votes)
print(isinstance(tallies, dict))
print(tallies["z"])
print(tallies.most_common(2))
print(tallies["a"])

True
0
[('a', 3), ('b', 1)]
3


## When to use it

Use `Counter` for frequencies, top-n, and merging tallies. Use a plain dict when a missing key should be an error. Don't use `Counter` as a general defaulting dict for lists or sets. That is `defaultdict`.


## Common mistake

Treating a missing key like a dict and wrapping every access in `if key in counts`. The zero default already handles the first sighting, so the guard is noise and hides the point of `Counter`.


In [4]:
from collections import Counter

counts = Counter()
for letter in "aab":
    if letter not in counts:
        counts[letter] = 0
    counts[letter] += 1
print(counts)

Counter({'a': 2, 'b': 1})


Drop the guard. `counts[letter] += 1` works on the first sighting because a missing key reads as `0`.


In [5]:
from collections import Counter

counts = Counter()
for letter in "aab":
    counts[letter] += 1
print(counts)

Counter({'a': 2, 'b': 1})


## Try it

Change `winner` so it returns the second-most-common name when there are at least two distinct votes. What does `most_common(2)` give you on a single-item list?


## Recap

- `Counter` is a dict subclass built for counts.
- A missing key returns `0`.
- `most_common` ranks by count.

**Next:** [practice](01-counter-practice.ipynb)
